In [ ]:
import random
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/all-MiniLM-L6-v2"
dataset_name = "glue"
dataset_config = "stsb"
preferred_split = "test"
fallback_split = "train"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 128 if device == "mps" else 64
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "preferred_split": preferred_split,
    "fallback_split": fallback_split,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})

In [ ]:
candidate_splits = [preferred_split, fallback_split, "validation"]
df = None
split_name = None
label_column_used = None

for candidate_split in candidate_splits:
    try:
        ds = load_dataset(dataset_name, dataset_config, split=candidate_split)
        candidate_df = ds.to_pandas().copy()
    except Exception as e:
        print({"split": candidate_split, "load_status": "failed", "error": str(e)[:300]})
        continue

    required_text_cols = {"sentence1", "sentence2"}
    if not required_text_cols.issubset(candidate_df.columns):
        print({"split": candidate_split, "load_status": "missing_text_columns", "columns": candidate_df.columns.tolist()})
        continue

    possible_label_cols = ["label", "score", "similarity_score"]
    found_label_col = None
    for col in possible_label_cols:
        if col in candidate_df.columns:
            numeric_values = pd.to_numeric(candidate_df[col], errors="coerce")
            if numeric_values.notna().any():
                found_label_col = col
                candidate_df[col] = numeric_values.astype(np.float32)
                break

    if found_label_col is None:
        print({"split": candidate_split, "load_status": "no_accessible_labels", "columns": candidate_df.columns.tolist()})
        continue

    candidate_df = candidate_df[["sentence1", "sentence2", found_label_col]].copy()
    candidate_df = candidate_df.rename(columns={found_label_col: "label"})
    candidate_df = candidate_df.dropna(subset=["sentence1", "sentence2", "label"]).reset_index(drop=True)

    if len(candidate_df) == 0:
        print({"split": candidate_split, "load_status": "empty_after_filtering"})
        continue

    df = candidate_df
    split_name = candidate_split
    label_column_used = found_label_col
    break

if df is None:
    raise RuntimeError("Could not load a GLUE STS-B split with accessible labels.")

print({
    "selected_split": split_name,
    "label_column_used": label_column_used,
    "num_examples": int(len(df)),
    "columns": df.columns.tolist(),
})
print(df.head())

In [ ]:
model = SentenceTransformer(model_name, device=device)
model.eval()
print(model_name)

In [ ]:
sentences1 = df["sentence1"].tolist()
sentences2 = df["sentence2"].tolist()
labels = df["label"].to_numpy(dtype=np.float32)

emb1 = model.encode(
    sentences1,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

emb2 = model.encode(
    sentences2,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

cosine_similarity = np.sum(emb1 * emb2, axis=1).astype(np.float32)
predicted_score_0_5 = (2.5 * (cosine_similarity + 1.0)).astype(np.float32)
absolute_error = np.abs(predicted_score_0_5 - labels).astype(np.float32)
squared_error = np.square(predicted_score_0_5 - labels).astype(np.float32)
signed_error = (predicted_score_0_5 - labels).astype(np.float32)
agreement_gap = np.abs(cosine_similarity - (labels / 2.5 - 1.0)).astype(np.float32)

results_df = df.copy()
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["absolute_error"] = absolute_error
results_df["squared_error"] = squared_error
results_df["signed_error"] = signed_error
results_df["agreement_gap"] = agreement_gap

print(results_df[["sentence1", "sentence2", "label", "cosine_similarity", "predicted_score_0_5", "absolute_error"]].head(10))

In [ ]:
pearson_corr = pearsonr(results_df["predicted_score_0_5"], results_df["label"]).statistic
spearman_corr = spearmanr(results_df["predicted_score_0_5"], results_df["label"]).statistic
mae = float(results_df["absolute_error"].mean())
rmse = float(np.sqrt(results_df["squared_error"].mean()))

quantile_edges = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
quantile_labels = ["Q1_easiest", "Q2", "Q3", "Q4", "Q5_hardest"]

results_df["error_quantile_bucket"] = pd.qcut(
    results_df["absolute_error"],
    q=quantile_edges,
    labels=quantile_labels,
    duplicates="drop",
)

quantile_agg = (
    results_df.groupby("error_quantile_bucket", observed=False)
    .agg(
        count=("label", "size"),
        label_mean=("label", "mean"),
        pred_mean=("predicted_score_0_5", "mean"),
        cosine_mean=("cosine_similarity", "mean"),
        mae=("absolute_error", "mean"),
        rmse=("squared_error", lambda x: float(np.sqrt(np.mean(x)))),
        signed_error_mean=("signed_error", "mean"),
        min_absolute_error=("absolute_error", "min"),
        max_absolute_error=("absolute_error", "max"),
    )
    .reset_index()
)

error_quantiles = results_df["absolute_error"].quantile(quantile_edges).astype(float)

print({
    "pearson_correlation": round(float(pearson_corr), 6),
    "spearman_correlation": round(float(spearman_corr), 6),
    "mae_0_5": round(mae, 6),
    "rmse_0_5": round(rmse, 6),
})
print({"absolute_error_quantiles": {str(k): round(v, 6) for k, v in error_quantiles.to_dict().items()}})
print(quantile_agg)

In [ ]:
pd.set_option("display.max_colwidth", 160)

easiest_examples = (
    results_df.sort_values(
        by=["absolute_error", "agreement_gap", "label"],
        ascending=[True, True, False],
    )
    [["sentence1", "sentence2", "label", "predicted_score_0_5", "cosine_similarity", "absolute_error", "signed_error", "error_quantile_bucket"]]
    .head(12)
    .reset_index(drop=True)
)

hardest_examples = (
    results_df.sort_values(
        by=["absolute_error", "agreement_gap", "label"],
        ascending=[False, False, False],
    )
    [["sentence1", "sentence2", "label", "predicted_score_0_5", "cosine_similarity", "absolute_error", "signed_error", "error_quantile_bucket"]]
    .head(12)
    .reset_index(drop=True)
)

print("EASIEST_EXAMPLES")
print(easiest_examples)
print("\nHARDEST_EXAMPLES")
print(hardest_examples)

In [ ]:
runtime_seconds = time.time() - start_time

summary = {
    "device_used": device,
    "model_name": model_name,
    "dataset_split": f"{dataset_name}/{dataset_config}/{split_name}",
    "num_examples": int(len(df)),
    "pearson_correlation": round(float(pearson_corr), 6),
    "spearman_correlation": round(float(spearman_corr), 6),
    "mae_0_5": round(mae, 6),
    "rmse_0_5": round(rmse, 6),
    "runtime_seconds": round(float(runtime_seconds), 2),
}

print(summary)